In [1]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import LSTM, Dense, Dropout, GlobalAveragePooling2D,Input,Bidirectional,Lambda,TimeDistributed,Conv2D,Flatten
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

2025-04-24 00:16:25.934490: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 00:16:25.966897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745442986.007273   40612 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745442986.019065   40612 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 00:16:26.061208: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
FRAME_COUNT = 16
IMAGE_SIZE = 64
NUM_CLASSES = 3  
FOLDER_PATH = "../../img/ote"

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
base_model.trainable = False  
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=GlobalAveragePooling2D()(base_model.output))

def load_data_transfer_learning_from_new(folder, frame_count=FRAME_COUNT, img_size=(IMAGE_SIZE,IMAGE_SIZE), num_classes=NUM_CLASSES):
    X = []
    y = []

    for video_folder in sorted(os.listdir(folder)):
        video_path = os.path.join(folder, video_folder)
        if not os.path.isdir(video_path):
            continue

        try:
            _, label = video_folder.split('_')
            label = int(label[1])
        except ValueError:
            print(f"Skipping {video_folder}, invalid format")
            continue

        frames = []
        frame_files = sorted(os.listdir(video_path))[:frame_count]

        for frame_file in frame_files:
            frame_path = os.path.join(video_path, frame_file)
            img = cv2.imread(frame_path)
            if img is not None:
                img = cv2.resize(img, img_size)
                img = img / 255.0
                frames.append(img)

        if len(frames) == frame_count:
            features = feature_extractor.predict(np.array(frames), verbose=0)
            X.append(features)
            y.append(label)
        else:
            print(f"Skipping {video_folder}: only {len(frames)} frames found (needs {frame_count})")

    X = np.array(X)
    y = to_categorical(np.array(y), num_classes=num_classes)

    return X, y



X, y = load_data_transfer_learning_from_new(FOLDER_PATH, frame_count=FRAME_COUNT, img_size=(IMAGE_SIZE,IMAGE_SIZE), num_classes=NUM_CLASSES)

print("X shape:", X.shape) 
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



2025-04-24 00:16:32.589777: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Skipping 101_000: only 0 frames found (needs 16)
Skipping 125_100: only 0 frames found (needs 16)
Skipping 126_100: only 0 frames found (needs 16)
Skipping 127_000: only 0 frames found (needs 16)
Skipping 128_200: only 0 frames found (needs 16)
X shape: (36, 16, 2048)
y shape: (36, 3)


In [3]:
print(f"Test size: {X_test.shape[0]}")
print(f"Train size: {X_train.shape[0]}")

Test size: 8
Train size: 28


In [4]:
model=Sequential([
    Input(shape=(FRAME_COUNT, 2048)),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(32, return_sequences=False)),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(optimizer="RMSprop", loss='categorical_crossentropy', metrics=['accuracy'])

In [5]:
model.fit(X_train,y_train,batch_size=8,epochs=20,validation_split=0.1)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 896ms/step - accuracy: 0.5678 - loss: 0.8489 - val_accuracy: 1.0000 - val_loss: 0.0629
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 162ms/step - accuracy: 0.8985 - loss: 0.4153 - val_accuracy: 1.0000 - val_loss: 0.7147
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - accuracy: 0.8652 - loss: 0.6865 - val_accuracy: 1.0000 - val_loss: 0.2247
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step - accuracy: 0.8152 - loss: 0.5726 - val_accuracy: 1.0000 - val_loss: 0.1409
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step - accuracy: 0.8652 - loss: 0.4963 - val_accuracy: 1.0000 - val_loss: 0.1281
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step - accuracy: 0.8402 - loss: 0.5364 - val_accuracy: 1.0000 - val_loss: 0.1189
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.8527 - loss: 0.5341 - val_accuracy: 1.0000 - val_loss: 0.1230
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 171ms/step - accuracy: 0.8152 - loss: 0.6032 - val_accuracy: 1.0000 - val_loss

In [6]:
model.evaluate(X_test,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 1.0000 - loss: 0.1796


[0.17957703769207, 1.0]